In [1]:
!python3 --version

Python 3.11.11


In [2]:
!nvidia-smi

Fri Jun  6 16:42:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
import tensorflow as tf

from tensorflow import keras

print(tf.__version__)
print(sys.version_info)
for module in mpl, np, pd, sklearn, tf, keras:
    print(module.__name__, module.__version__)

2025-06-06 16:42:41.232003: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749228161.433331      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749228161.492181      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2.18.0
sys.version_info(major=3, minor=11, micro=11, releaselevel='final', serial=0)
matplotlib 3.7.2
numpy 1.26.4
pandas 2.2.3
sklearn 1.2.2
tensorflow 2.18.0
keras._tf_keras.keras 3.8.0


In [4]:
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"Num GPUs Available: {len(gpus)}")

if gpus:
    print(f"GPUs available: {gpus}")
    try:
        # 打印每个GPU的详细信息
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True) # 推荐设置，按需分配显存
            print(f"Details for {gpu.name}:")
            # 可以尝试执行一个小操作来确认
        print("GPU is available and TensorFlow can see it!")

        # 尝试一个简单的GPU运算
        print("\nAttempting a simple computation on GPU...")
        with tf.device('/GPU:0'): # 明确指定在第一个GPU上运行
            a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            b = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            c = tf.matmul(a, b)
        print("Matrix multiplication result from GPU:")
        print(c.numpy())
        print("If no errors occurred, GPU is working!")

    except RuntimeError as e:
        print(f"RuntimeError during GPU setup or test: {e}")
else:
    print("GPU not available to TensorFlow. TensorFlow will run on CPU.")

TensorFlow Version: 2.18.0
Num GPUs Available: 2
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Details for /physical_device:GPU:0:
Details for /physical_device:GPU:1:
GPU is available and TensorFlow can see it!

Attempting a simple computation on GPU...
Matrix multiplication result from GPU:
[[ 7. 10.]
 [15. 22.]]
If no errors occurred, GPU is working!


I0000 00:00:1749228173.774443      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1749228173.775110      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
# constant是常量张量
t = tf.constant([[1., 2., 3.], [4., 5.,6.]])

# index
#2.0能够直接获取值时因为execution默认打开的
print(t)
print(t[:, 1:])
print(t[..., 1])

tf.Tensor(
[[1. 2. 3.]
 [4. 5. 6.]], shape=(2, 3), dtype=float32)
tf.Tensor(
[[2. 3.]
 [5. 6.]], shape=(2, 2), dtype=float32)
tf.Tensor([2. 5.], shape=(2,), dtype=float32)


In [6]:
# t.assign(1) 对常量不能进行再次assign设置
type(t.numpy())  #转为ndarray
q=t.numpy()
t1= tf.constant(q)  # 把 ndarray 变为张量
t1

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)>

In [7]:
# ops 使用tf本身的math接口对Tensor进行计算
print(t+10)
print(t)
print(tf.square(t))
print(t)

tf.Tensor(
[[11. 12. 13.]
 [14. 15. 16.]], shape=(2, 3), dtype=float32)
tf.Tensor(
[[1. 2. 3.]
 [4. 5. 6.]], shape=(2, 3), dtype=float32)
tf.Tensor(
[[ 1.  4.  9.]
 [16. 25. 36.]], shape=(2, 3), dtype=float32)
tf.Tensor(
[[1. 2. 3.]
 [4. 5. 6.]], shape=(2, 3), dtype=float32)


In [8]:
# 矩阵乘以自己的转置
print(tf.transpose(t))
print(t @ tf.transpose(t))  # @是矩阵乘法，和*不一致

tf.Tensor(
[[1. 4.]
 [2. 5.]
 [3. 6.]], shape=(3, 2), dtype=float32)
tf.Tensor(
[[14. 32.]
 [32. 77.]], shape=(2, 2), dtype=float32)


In [9]:
print(tf.sqrt(t))
print('-'*50)
# tf.math.sqrt(t)   这个更规范
tf.math.log(t)  # 必须加 math

tf.Tensor(
[[1.        1.4142135 1.7320508]
 [2.        2.2360678 2.4494896]], shape=(2, 3), dtype=float32)
--------------------------------------------------


<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[0.       , 0.6931472, 1.0986123],
       [1.3862944, 1.609438 , 1.7917595]], dtype=float32)>

In [10]:
# numpy conversion
print(t.numpy())  #可以直接通过numpy取出来
print(t.numpy().tolist())
print(type(t.numpy()))

[[1. 2. 3.]
 [4. 5. 6.]]
[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
<class 'numpy.ndarray'>


In [11]:
print(np.square(t)) #直接求平方
np_t = np.array([[1., 2., 3.], [4., 5., 6.]])
print(tf.constant(np_t)) #转换为tensor

[[ 1.  4.  9.]
 [16. 25. 36.]]
tf.Tensor(
[[1. 2. 3.]
 [4. 5. 6.]], shape=(2, 3), dtype=float64)


In [12]:
# Scalars  就是标量，只有一个数值的张量，称为标量
t = tf.constant(2.718)
print(t)
print(t.numpy())
print(t.shape)  #维数

tf.Tensor(2.718, shape=(), dtype=float32)
2.718
()


In [13]:
# strings
t = tf.constant("cafe")
print(t)
print(tf.strings.length(t))
print(tf.strings.length(t, unit="UTF8_CHAR"))
print(tf.strings.unicode_decode(t, "UTF8"))

tf.Tensor(b'cafe', shape=(), dtype=string)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor([ 99  97 102 101], shape=(4,), dtype=int32)


In [14]:
# string array
t = tf.constant(["cafe", "coffee", "咖啡"])
#自动求出数组中每一个字符的长度,如果不加unit="UTF8_CHAR"，得到的是实际字节存储的长度
print(tf.strings.length(t, unit="UTF8_CHAR"))  
print(tf.strings.length(t, unit="BYTE"))  
r = tf.strings.unicode_decode(t, "UTF8")
# https://tool.chinaz.com/tools/unicode.aspx  汉字转的是unicode编码
print(r)

tf.Tensor([4 6 2], shape=(3,), dtype=int32)
tf.Tensor([4 6 6], shape=(3,), dtype=int32)
<tf.RaggedTensor [[99, 97, 102, 101], [99, 111, 102, 102, 101, 101], [21654, 21857]]>


RaggedTensor 是指形状分布不固定的（行元素个数不相等）。

实际深度学习里不支持这种形状不规则的 tensor 的输入，**这样的不规则 tensor 的目的是节省空间**。

In [15]:
# ragged tensor
r = tf.ragged.constant([[11, 12], [21, 22, 23], [], [41]])

# index op
print(r)
print(r.shape)
print(r[1])
#取一行也是ragged tensor
print(r[1:3])     # 左闭右开
# print(r[:,1])#不能取列索引

<tf.RaggedTensor [[11, 12], [21, 22, 23], [], [41]]>
(4, None)
tf.Tensor([21 22 23], shape=(3,), dtype=int32)
<tf.RaggedTensor [[21, 22, 23], []]>


In [16]:
# ops on ragged tensor
r2 = tf.ragged.constant([[51, 52],[], [], [71]])
print(tf.concat([r, r2], axis = 0))
print(tf.concat([r, r2], axis = 1))  # 如果行数不相等的话，不可以拼

<tf.RaggedTensor [[11, 12], [21, 22, 23], [], [41], [51, 52], [], [], [71]]>
<tf.RaggedTensor [[11, 12, 51, 52], [21, 22, 23], [], [41, 71]]>


各种深度学习模型必须输入一个tensor，空闲的补0，只能往后面补。

In [17]:
print(r.to_tensor())

tf.Tensor(
[[11 12  0]
 [21 22 23]
 [ 0  0  0]
 [41  0  0]], shape=(4, 3), dtype=int32)


稀疏矩阵：

In [18]:
# sparse tensor 可以往前面补零,sparse tensor从第一行依次往下填位置
# sparse tensor 存储节省内存空间，磁盘空间
s = tf.SparseTensor(indices = [[0, 1], [1, 0], [2, 3],[3,2]], # 位置
                    values = [1., 2., 3.,5],  # 值
                    dense_shape = [4, 4])  # 维数
print(s)
tt=tf.sparse.to_dense(s)
tt

SparseTensor(indices=tf.Tensor(
[[0 1]
 [1 0]
 [2 3]
 [3 2]], shape=(4, 2), dtype=int64), values=tf.Tensor([1. 2. 3. 5.], shape=(4,), dtype=float32), dense_shape=tf.Tensor([4 4], shape=(2,), dtype=int64))


<tf.Tensor: shape=(4, 4), dtype=float32, numpy=
array([[0., 1., 0., 0.],
       [2., 0., 0., 0.],
       [0., 0., 0., 3.],
       [0., 0., 5., 0.]], dtype=float32)>

In [19]:
# ops on sparse tensors
s2 = s * 2.0
print(s2)

SparseTensor(indices=tf.Tensor(
[[0 1]
 [1 0]
 [2 3]
 [3 2]], shape=(4, 2), dtype=int64), values=tf.Tensor([ 2.  4.  6. 10.], shape=(4,), dtype=float32), dense_shape=tf.Tensor([4 4], shape=(2,), dtype=int64))


In [20]:
#不支持加法
try:
    s3 = s + 1
except TypeError as ex:
    print(ex)

unsupported operand type(s) for +: 'SparseTensor' and 'int'


In [21]:
s4 = tf.constant([[10., 20.],
                  [30., 40.],
                  [50., 60.],
                  [70., 80.]])
# tf.sparse.to_dense(s)@s4
print(tf.sparse.sparse_dense_matmul(s, s4)) #稀疏Tensor和Tensor想乘

tf.Tensor(
[[ 30.  40.]
 [ 20.  40.]
 [210. 240.]
 [250. 300.]], shape=(4, 2), dtype=float32)


sparse 无顺序时，不能转为tensor，会报错：

In [22]:
# sparse tensor
s5 = tf.SparseTensor(indices = [[0, 2], [2, 3], [0, 1]],
                    values = [1., 2., 3.],
                    dense_shape = [3, 4])
# print(tf.sparse.to_dense(s5))  #sparse无顺序时，不能转为tensor，会报错
print(s5)
s6 = tf.sparse.reorder(s5)
print(s6)
print(tf.sparse.to_dense(s6))

SparseTensor(indices=tf.Tensor(
[[0 2]
 [2 3]
 [0 1]], shape=(3, 2), dtype=int64), values=tf.Tensor([1. 2. 3.], shape=(3,), dtype=float32), dense_shape=tf.Tensor([3 4], shape=(2,), dtype=int64))
SparseTensor(indices=tf.Tensor(
[[0 1]
 [0 2]
 [2 3]], shape=(3, 2), dtype=int64), values=tf.Tensor([3. 1. 2.], shape=(3,), dtype=float32), dense_shape=tf.Tensor([3 4], shape=(2,), dtype=int64))
tf.Tensor(
[[0. 3. 1. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 2.]], shape=(3, 4), dtype=float32)


In [23]:
# Variables
v = tf.Variable([[1., 2., 3.], [4., 5.,6.]])
print(v)
print(v.value())
print('-'*50)
print(v.numpy())

<tf.Variable 'Variable:0' shape=(2, 3) dtype=float32, numpy=
array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)>
tf.Tensor(
[[1. 2. 3.]
 [4. 5. 6.]], shape=(2, 3), dtype=float32)
--------------------------------------------------
[[1. 2. 3.]
 [4. 5. 6.]]


In [24]:
# 修改变量时要用assign，改变tensor内某个值，空间没有发生变化，效率高
# assign value
print(id(v))
v.assign(2*v)
print(id(v))
print(v.numpy())
print('-'*50)
v[0, 1].assign(42)  #取某个元素修改
print(v.numpy())
print('-'*50)
v[1].assign([7., 8., 9.])  #取某一行修改
print(v.numpy())
print(id(v))

138921488294736
138921488294736
[[ 2.  4.  6.]
 [ 8. 10. 12.]]
--------------------------------------------------
[[ 2. 42.  6.]
 [ 8. 10. 12.]]
--------------------------------------------------
[[ 2. 42.  6.]
 [ 7.  8.  9.]]
138921488294736


In [25]:
try:
    v[1] = [7., 8., 9.]
except TypeError as ex:
    print(ex)

'ResourceVariable' object does not support item assignment


In [26]:
v=2*v
print(v)
print(id(v))
print(type(v))

tf.Tensor(
[[ 4. 84. 12.]
 [14. 16. 18.]], shape=(2, 3), dtype=float32)
138919804385040
<class 'tensorflow.python.framework.ops.EagerTensor'>


In [27]:
x = tf.constant([[1., 1.], [2., 2.]])
tf.reduce_mean(x,axis=1)

<tf.Tensor: shape=(2,), dtype=float32, numpy=array([1., 2.], dtype=float32)>